In [3]:
!pip install lightgbm

  Obtaining dependency information for lightgbm from https://files.pythonhosted.org/packages/42/86/dabda8fbcb1b00bcfb0003c3776e8ade1aa7b413dff0a2c08f457dace22f/lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 10.7 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [11]:
import os
import pickle

import numpy as np
import pandas as pd
import sklearn
from scipy.stats import norm
import lightgbm as lgb
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split

In [12]:
# データの読み込み
ROOT_DIR = "../dataset"

all_train_df = pd.read_pickle(f"{ROOT_DIR}/0109_preprocessed_train_df.pkl")

In [13]:
all_train_df["target_ym"].value_counts()

target_ym
202007    45075
202001    44206
202207    41236
202101    39475
201907    38659
202201    37834
202107    37093
201901    30086
Name: count, dtype: int64

In [14]:
all_train_df.columns

Index(['addr1_1', 'addr1_2', 'building_type', 'building_structure',
       'bukken_id', 'convenience_distance', 'drugstore_distance', 'eki_name1',
       'eki_name2', 'floor_count', 'house_area', 'lat', 'lon',
       'madori_kind_all', 'madori_number_all', 'money_kyoueki',
       'money_kyoueki_tax', 'money_rimawari_now', 'park_distance',
       'parking_distance', 'post1', 'post2', 'room_kaisuu', 'rosen_name1',
       'rosen_name2', 'school_ele_distance', 'school_jun_distance',
       'target_ym', 'unit_area', 'year_built', 'walk_distance1',
       'walk_distance2', 'money_room', 'target_y', 'madori_str', 'post_number',
       'station_count_800m', 'total_passengers_800m', 'station_count_1600m',
       'total_passengers_1600m', 'land_value_median_5nn',
       'land_value_ratio_mean_5nn'],
      dtype='object')

In [15]:
all_train_df.drop("bukken_id", axis=1, inplace=True)

In [16]:
NUMERICAL_FEATURES = [
    "convenience_distance",
    "floor_count",
    "house_area",
    "land_kenpei",
    'lat', 'lon',
    'money_kyoueki',
    'money_kyoueki_tax',
    'money_rimawari_now',
    'park_distance',
    'parking_distance',
    'school_ele_distance',
    'school_jun_distance',
    'walk_distance1',
    'walk_distance2',
    'money_room',
    'station_count_800m',
    'total_passengers_800m',
    'station_count_1600m',
    'total_passengers_1600m',
    'target_y',
    "land_value_median_5nn",
    "land_value_ratio_mean_5nn"
]

In [17]:
def prepare_dataset_per_sqm(df_train, df_valid):
    # 平米単価の計算
    df_train['house_area'] = df_train['house_area'].astype(float)
    df_valid['house_area'] = df_valid['house_area'].astype(float)

    if 'price_per_sqm' not in df_train.columns:
        df_train['price_per_sqm'] = df_train['money_room'].astype(float) / df_train['house_area']
    if 'price_per_sqm' not in df_valid.columns:
        df_valid['price_per_sqm'] = df_valid['money_room'].astype(float) / df_valid['house_area']

    # 説明変数の分離
    drop_cols = ['money_room', 'price_per_sqm', 'target_ym']
    df_x_train = df_train.drop(drop_cols, axis=1, errors='ignore').copy()
    df_x_valid = df_valid.drop(drop_cols, axis=1, errors='ignore').copy()

    #平米単価を対数変換
    df_y_train = np.log1p(df_train['price_per_sqm'].astype(float))
    df_y_valid_log = np.log1p(df_valid['price_per_sqm'].astype(float))

    # エンコード処理
    label_encoder = LabelEncoder()
    encode_cols = list(set(df_x_train.columns.to_list()) - set(drop_cols) - set(NUMERICAL_FEATURES))

    for col in encode_cols:
        if col in df_x_train.columns:
            all_values = pd.concat([df_x_train[col], df_x_valid[col]]).astype(str)
            label_encoder.fit(all_values)
            df_x_train[col] = label_encoder.transform(df_x_train[col].astype(str))
            df_x_valid[col] = label_encoder.transform(df_x_valid[col].astype(str))

    # 数値変換
    for col in NUMERICAL_FEATURES:
        if col in df_x_train.columns:
            df_x_train[col] = df_x_train[col].astype(float)
            df_x_valid[col] = df_x_valid[col].astype(float)

    # カラム整合性確保
    common_cols = [c for c in df_x_train.columns if c in df_x_valid.columns]
    df_x_train = df_x_train[common_cols]
    df_x_valid = df_x_valid[common_cols]

    return df_x_train, df_y_train, df_x_valid, df_y_valid_log


#平米単価予測 + Log + Fair Loss モデル
def train_and_predict_per_sqm_log_fair(df_train, df_valid, params, num_iter, model_name="PerSqm_LogFair"):
    df_train = df_train.copy()
    df_valid = df_valid.copy()

    df_x_train, df_y_train, df_x_valid, df_y_valid_log = prepare_dataset_per_sqm(df_train, df_valid)

    if len(df_x_train) == 0 or len(df_x_valid) == 0:
        print(f"[{model_name}] データ不足のためスキップ")
        return None, None

    # LightGBMデータセット作成
    lgb_train = lgb.Dataset(df_x_train, df_y_train)
    lgb_eval = lgb.Dataset(df_x_valid, df_y_valid_log, reference=lgb_train)

    # 学習
    model = lgb.train(
        params,
        lgb_train,
        valid_sets=[lgb_train, lgb_eval],
        valid_names=['train', 'eval'],
        num_boost_round=num_iter,
        callbacks=[lgb.early_stopping(stopping_rounds=500, verbose=True), lgb.log_evaluation(500)]
    )

    # 予測(対数空間の平米単価)
    y_pred_log_sqm = model.predict(df_x_valid)

    # 対数から元の平米単価に戻す
    y_pred_sqm = np.expm1(y_pred_log_sqm)

    # 平米単価 × 面積 で総額に変換
    y_pred_total = y_pred_sqm * df_x_valid['house_area']
    y_true_total = df_valid['money_room'].astype(float)

    # 実金額ベースでのMAPE計算
    mape = mean_absolute_percentage_error(y_true_total, y_pred_total)
    print(f"[{model_name}] MAPE: {mape * 100:.2f}% (件数: {len(y_true_total)})")

    return model,y_true_total, y_pred_total


def prepare_dataset_direct(df_train, df_valid):
        # 説明変数の分離
    drop_cols = ['money_room', 'target_ym', 'price_per_sqm']
    df_x_train = df_train.drop(drop_cols, axis=1, errors='ignore').copy()
    df_x_valid = df_valid.drop(drop_cols, axis=1, errors='ignore').copy()

    # 目的変数を対数変換
    df_y_train = np.log1p(df_train['money_room'].astype(float))
    df_y_valid_log = np.log1p(df_valid['money_room'].astype(float))

    # エンコード処理
    label_encoder = LabelEncoder()
    encode_cols = list(set(df_x_train.columns.to_list()) - set(drop_cols) - set(NUMERICAL_FEATURES))

    for col in encode_cols:
        if col in df_x_train.columns:
            all_values = pd.concat([df_x_train[col], df_x_valid[col]]).astype(str)
            label_encoder.fit(all_values)
            df_x_train[col] = label_encoder.transform(df_x_train[col].astype(str))
            df_x_valid[col] = label_encoder.transform(df_x_valid[col].astype(str))

    # 数値変換
    for col in NUMERICAL_FEATURES:
        if col in df_x_train.columns:
            df_x_train[col] = df_x_train[col].astype(float)
            df_x_valid[col] = df_x_valid[col].astype(float)

    # カラムの整合性確保
    common_cols = [c for c in df_x_train.columns if c in df_x_valid.columns]
    df_x_train = df_x_train[common_cols]
    df_x_valid = df_x_valid[common_cols]

    return df_x_train, df_y_train, df_x_valid, df_y_valid_log


def train_and_predict_direct_log_fair(df_train, df_valid, params, num_iter, model_name="Direct_LogFair"):
    df_train = df_train.copy()
    df_valid = df_valid.copy()

    df_x_train, df_y_train, df_x_valid, df_y_valid_log = prepare_dataset_direct(df_train, df_valid)

    if len(df_x_train) == 0 or len(df_x_valid) == 0:
        print(f"[{model_name}] データ不足のためスキップ")
        return None, None
    
    # LightGBMデータセット作成
    lgb_train = lgb.Dataset(df_x_train, df_y_train)
    lgb_eval = lgb.Dataset(df_x_valid, df_y_valid_log, reference=lgb_train)

    # 学習
    model = lgb.train(
        params,
        lgb_train,
        valid_sets=[lgb_train, lgb_eval],
        valid_names=['train', 'eval'],
        num_boost_round=num_iter,
        callbacks=[lgb.early_stopping(stopping_rounds=500, verbose=True),lgb.log_evaluation(500)]
    )

    # 予測
    y_pred_log = model.predict(df_x_valid)

    # 対数から元の金額に戻す
    y_pred = np.expm1(y_pred_log)

    # 正解データ
    y_true = df_valid['money_room'].astype(float)

    # 実金額ベースでのMAPE計算
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"[{model_name}] MAPE: {mape * 100:.2f}% (件数: {len(y_true)})")

    return model, y_true, y_pred

## （東京近郊、沖縄、その他）✖︎建物別モデルの学習

In [18]:
# モデルを分ける地域
SPLITED_TOKYO = [11, 12, 13, 14] # 埼玉、千葉、東京、神奈川
SPLITED_OKINAWA = [47] # 沖縄

# # モデルを分ける建物
# SPLITED_BUILDING1 = "マンション"
# SPLITED_BUILDING2 = "一軒家"

In [19]:
# Fair Lossを使用したパラメータ設定
PARAMS = {
    'objective': 'fair',      # Fair Loss
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'fair_c': 0.1,
    'verbose': -1,
    'random_state': 42,
    "learning_rate": 0.05
}
NUM_ITER = 20000

In [20]:
# シャッフルでバリデーションセットを抽出
train_df, valid_df = train_test_split(all_train_df, test_size=0.2, random_state=42)

In [21]:
# モデル保存先の作成
SAVE_PATH = "../model/0109"
os.makedirs(SAVE_PATH, exist_ok=True)

In [22]:
# 地域で分割
# 東京
df_train_tokyo = train_df[train_df['addr1_1'].isin(SPLITED_TOKYO)]
df_valid_tokyo = valid_df[valid_df['addr1_1'].isin(SPLITED_TOKYO)]
# 沖縄
df_train_okinawa = train_df[train_df['addr1_1'].isin(SPLITED_OKINAWA)]
df_valid_okinawa = valid_df[valid_df['addr1_1'].isin(SPLITED_OKINAWA)]
# それ以外
df_train_other = train_df[~train_df['addr1_1'].isin(SPLITED_TOKYO + SPLITED_OKINAWA)]
df_valid_other = valid_df[~valid_df['addr1_1'].isin(SPLITED_TOKYO + SPLITED_OKINAWA)]

# 物件種別で分割
def split_by_type(df):
    is_mansion = df['building_type'].astype(str) == "マンション"
    return df[is_mansion].copy(), df[~is_mansion].copy()

# 東京近郊
df_train_tokyo_mansion, df_train_tokyo_others = split_by_type(df_train_tokyo)
df_valid_tokyo_mansion, df_valid_tokyo_others = split_by_type(df_valid_tokyo)
# 沖縄
df_train_okinawa_mansion, df_train_okinawa_others = split_by_type(df_train_okinawa)
df_valid_okinawa_mansion, df_valid_okinawa_others = split_by_type(df_valid_okinawa)
# それ以外
df_train_other_mansion, df_train_other_others = split_by_type(df_train_other)
df_valid_other_mansion, df_valid_other_others = split_by_type(df_valid_other) 

In [23]:
# 全体統合用のリスト
all_y_true = []
all_y_pred = []

# --- 以下、各エリア・種別ごとにモデル実行 ---
for area_name, train_m, test_m, train_o, test_o in [
    ("東京近郊", df_train_tokyo_mansion, df_valid_tokyo_mansion, df_train_tokyo_others, df_valid_tokyo_others),
    ("沖縄", df_train_okinawa_mansion, df_valid_okinawa_mansion, df_train_okinawa_others, df_valid_okinawa_others),
    ("その他地域", df_train_other_mansion, df_valid_other_mansion, df_train_other_others, df_valid_other_others)
]:
    print(f"\n{'='*60}\n【{area_name}】平米単価(Log+Fair) × 物件種別モデル\n{'='*60}")

    print("\n=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===")
    model_name = f"{area_name}_Mansion_PerSqm_LogFair_{NUM_ITER}"
    model, y_true_m, y_pred_m = train_and_predict_per_sqm_log_fair(
        train_m, test_m, PARAMS, NUM_ITER, model_name=model_name
    )
    # モデルの保存
    with open(f"{SAVE_PATH}/{model_name}.pkl", "wb") as f:
        pickle.dump(model, f)

    print("\n=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===")
    model_name = f"{area_name}_Others_Direct_LogFair_{NUM_ITER}"
    model, y_true_o, y_pred_o = train_and_predict_direct_log_fair(
        train_o, test_o, PARAMS, NUM_ITER, model_name=model_name
    )
    # モデルの保存
    with open(f"{SAVE_PATH}/{model_name}.pkl", "wb") as f:
        pickle.dump(model, f)

    # 結果の統合（エリア内）
    if y_true_m is not None and y_true_o is not None:
        y_true_all = pd.concat([y_true_m, y_true_o])
        y_pred_all = np.concatenate([y_pred_m, y_pred_o])

        # 全体統合用に追加
        all_y_true.append(y_true_all)
        all_y_pred.append(y_pred_all)

        total_mape = mean_absolute_percentage_error(y_true_all, y_pred_all)
        print("\n" + "="*60)
        print(f"[{area_name}] 統合モデルの全体MAPE: {total_mape * 100:.2f}%")
        print(f"  - マンション件数: {len(y_true_m)}")
        print(f"  - 戸建・他件数: {len(y_true_o)}")
        print(f"  - 合計件数: {len(y_true_all)}")
        print("="*60)
    else:
        print(f"\n[警告] {area_name}でいずれかのモデルでデータ不足のため統合できませんでした")

# --- 全エリア・全種別を統合した最終MAPE ---
if len(all_y_true) > 0:
    final_y_true = pd.concat(all_y_true)
    final_y_pred = np.concatenate(all_y_pred)

    final_mape = mean_absolute_percentage_error(final_y_true, final_y_pred)

    print("\n" + "="*60)
    print("【最終結果】全エリア・全種別統合モデル")
    print("="*60)
    print(f"全体MAPE: {final_mape * 100:.2f}%")
    print(f"総件数: {len(final_y_true)}")
    print("="*60)


【東京近郊】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
Training until validation scores don't improve for 500 rounds
[500]	train's rmse: 0.161781	eval's rmse: 0.176475
[1000]	train's rmse: 0.144786	eval's rmse: 0.169521
[1500]	train's rmse: 0.133779	eval's rmse: 0.166186
[2000]	train's rmse: 0.125601	eval's rmse: 0.164255
[2500]	train's rmse: 0.118728	eval's rmse: 0.162706
[3000]	train's rmse: 0.112941	eval's rmse: 0.161563
[3500]	train's rmse: 0.108038	eval's rmse: 0.160745
[4000]	train's rmse: 0.103594	eval's rmse: 0.160033
[4500]	train's rmse: 0.0995415	eval's rmse: 0.159499
[5000]	train's rmse: 0.0960073	eval's rmse: 0.159022
[5500]	train's rmse: 0.0929277	eval's rmse: 0.158631
[6000]	train's rmse: 0.0899728	eval's rmse: 0.158225
[6500]	train's rmse: 0.0871657	eval's rmse: 0.157915
[7000]	train's rmse: 0.0846851	eval's rmse: 0.157698
[7500]	train's rmse: 0.0822553	eval's rmse: 0.157454
[8000]	train's rmse: 0.0800752	eval's rmse: 0.157235
[8500]	train's rmse: 0.0

## （東京、郊外）✖︎建物別モデルの学習

In [24]:
# モデルを分ける地域
SPLITED_TOKYO = [13] # 東京
SPLITED_KOUGAI = [11, 12, 14] # 埼玉、千葉、神奈川
SPLITED_OKINAWA = [47] # 沖縄

# # モデルを分ける建物
# SPLITED_BUILDING1 = "マンション"
# SPLITED_BUILDING2 = "一軒家"

In [25]:
# Fair Lossを使用したパラメータ設定
PARAMS = {
    'objective': 'fair',      # Fair Loss
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'fair_c': 0.1,
    'verbose': -1,
    'random_state': 42,
    "learning_rate": 0.05
}
NUM_ITER = 20000

In [26]:
# シャッフルでバリデーションセットを抽出
train_df, valid_df = train_test_split(all_train_df, test_size=0.2, random_state=42)

In [27]:
# モデル保存先の作成
SAVE_PATH = "../model/0109_ver2"
os.makedirs(SAVE_PATH, exist_ok=True)

In [28]:
# 地域で分割
# 東京
df_train_tokyo = train_df[train_df['addr1_1'].isin(SPLITED_TOKYO)]
df_valid_tokyo = valid_df[valid_df['addr1_1'].isin(SPLITED_TOKYO)]
# 郊外
df_train_kougai = train_df[train_df['addr1_1'].isin(SPLITED_KOUGAI)]
df_valid_kougai = valid_df[valid_df['addr1_1'].isin(SPLITED_KOUGAI)]
# 沖縄
df_train_okinawa = train_df[train_df['addr1_1'].isin(SPLITED_OKINAWA)]
df_valid_okinawa = valid_df[valid_df['addr1_1'].isin(SPLITED_OKINAWA)]
# それ以外
df_train_other = train_df[~train_df['addr1_1'].isin(SPLITED_TOKYO + SPLITED_KOUGAI + SPLITED_OKINAWA)]
df_valid_other = valid_df[~valid_df['addr1_1'].isin(SPLITED_TOKYO + SPLITED_KOUGAI + SPLITED_OKINAWA)]

# 物件種別で分割
def split_by_type(df):
    is_mansion = df['building_type'].astype(str) == "マンション"
    return df[is_mansion].copy(), df[~is_mansion].copy()

# 東京
df_train_tokyo_mansion, df_train_tokyo_others = split_by_type(df_train_tokyo)
df_valid_tokyo_mansion, df_valid_tokyo_others = split_by_type(df_valid_tokyo)
# 郊外
df_train_kougai_mansion, df_train_kougai_others = split_by_type(df_train_kougai)
df_valid_kougai_mansion, df_valid_kougai_others = split_by_type(df_valid_kougai)
# 沖縄
df_train_okinawa_mansion, df_train_okinawa_others = split_by_type(df_train_okinawa)
df_valid_okinawa_mansion, df_valid_okinawa_others = split_by_type(df_valid_okinawa)
# それ以外
df_train_other_mansion, df_train_other_others = split_by_type(df_train_other)
df_valid_other_mansion, df_valid_other_others = split_by_type(df_valid_other) 

In [29]:
# 全体統合用のリスト
all_y_true = []
all_y_pred = []

# --- 以下、各エリア・種別ごとにモデル実行 ---
for area_name, train_m, test_m, train_o, test_o in [
    ("東京", df_train_tokyo_mansion, df_valid_tokyo_mansion, df_train_tokyo_others, df_valid_tokyo_others),
    ("東京近郊", df_train_kougai_mansion, df_valid_kougai_mansion, df_train_kougai_others, df_valid_kougai_others),
    ("沖縄", df_train_okinawa_mansion, df_valid_okinawa_mansion, df_train_okinawa_others, df_valid_okinawa_others),
    ("その他地域", df_train_other_mansion, df_valid_other_mansion, df_train_other_others, df_valid_other_others)
]:
    print(f"\n{'='*60}\n【{area_name}】平米単価(Log+Fair) × 物件種別モデル\n{'='*60}")

    print("\n=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===")
    model_name = f"{area_name}_Mansion_PerSqm_LogFair_{NUM_ITER}"
    model, y_true_m, y_pred_m = train_and_predict_per_sqm_log_fair(
        train_m, test_m, PARAMS, NUM_ITER, model_name=model_name
    )
    # モデルの保存
    with open(f"{SAVE_PATH}/{model_name}.pkl", "wb") as f:
        pickle.dump(model, f)

    print("\n=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===")
    model_name = f"{area_name}_Others_Direct_LogFair_{NUM_ITER}"
    model, y_true_o, y_pred_o = train_and_predict_direct_log_fair(
        train_o, test_o, PARAMS, NUM_ITER, model_name=model_name
    )
    # モデルの保存
    with open(f"{SAVE_PATH}/{model_name}.pkl", "wb") as f:
        pickle.dump(model, f)

    # 結果の統合（エリア内）
    if y_true_m is not None and y_true_o is not None:
        y_true_all = pd.concat([y_true_m, y_true_o])
        y_pred_all = np.concatenate([y_pred_m, y_pred_o])

        # 全体統合用に追加
        all_y_true.append(y_true_all)
        all_y_pred.append(y_pred_all)

        total_mape = mean_absolute_percentage_error(y_true_all, y_pred_all)
        print("\n" + "="*60)
        print(f"[{area_name}] 統合モデルの全体MAPE: {total_mape * 100:.2f}%")
        print(f"  - マンション件数: {len(y_true_m)}")
        print(f"  - 戸建・他件数: {len(y_true_o)}")
        print(f"  - 合計件数: {len(y_true_all)}")
        print("="*60)
    else:
        print(f"\n[警告] {area_name}でいずれかのモデルでデータ不足のため統合できませんでした")

# --- 全エリア・全種別を統合した最終MAPE ---
if len(all_y_true) > 0:
    final_y_true = pd.concat(all_y_true)
    final_y_pred = np.concatenate(all_y_pred)

    final_mape = mean_absolute_percentage_error(final_y_true, final_y_pred)

    print("\n" + "="*60)
    print("【最終結果】全エリア・全種別統合モデル")
    print("="*60)
    print(f"全体MAPE: {final_mape * 100:.2f}%")
    print(f"総件数: {len(final_y_true)}")
    print("="*60)


【東京】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
Training until validation scores don't improve for 500 rounds
[500]	train's rmse: 0.122586	eval's rmse: 0.144394
[1000]	train's rmse: 0.104875	eval's rmse: 0.138958
[1500]	train's rmse: 0.093205	eval's rmse: 0.13659
[2000]	train's rmse: 0.0844998	eval's rmse: 0.13527
[2500]	train's rmse: 0.0775112	eval's rmse: 0.134506
[3000]	train's rmse: 0.0716151	eval's rmse: 0.133881
[3500]	train's rmse: 0.0665346	eval's rmse: 0.133491
[4000]	train's rmse: 0.0623247	eval's rmse: 0.133187
[4500]	train's rmse: 0.0584553	eval's rmse: 0.133005
[5000]	train's rmse: 0.0550383	eval's rmse: 0.1329
[5500]	train's rmse: 0.051991	eval's rmse: 0.132837
[6000]	train's rmse: 0.0493378	eval's rmse: 0.132726
[6500]	train's rmse: 0.0468964	eval's rmse: 0.132625
[7000]	train's rmse: 0.0445884	eval's rmse: 0.132566
[7500]	train's rmse: 0.0424894	eval's rmse: 0.132524
[8000]	train's rmse: 0.0405462	eval's rmse: 0.132479
[8500]	train's rmse: 0.038

KeyboardInterrupt: 

### 精度確認

In [30]:
# 物件種別で分割
def split_by_type(df):
    is_mansion = df['building_type'].astype(str) == "マンション"
    return df[is_mansion].copy(), df[~is_mansion].copy()


def evaluate_mansion(model, df_train, df_valid, model_name):
    df_train = df_train.copy()
    df_valid = df_valid.copy()

    _, _, df_x_valid, _ = prepare_dataset_per_sqm(df_train, df_valid)
    # 予測(対数空間の平米単価)
    y_pred_log_sqm = model.predict(df_x_valid)

    # 対数から元の平米単価に戻す
    y_pred_sqm = np.expm1(y_pred_log_sqm)

    # 平米単価 × 面積 で総額に変換
    y_pred_total = y_pred_sqm * df_x_valid['house_area']
    y_true_total = df_valid['money_room'].astype(float)

    # 実金額ベースでのMAPE計算
    mape = mean_absolute_percentage_error(y_true_total, y_pred_total)
    print(f"[{model_name}] MAPE: {mape * 100:.2f}% (件数: {len(y_true_total)})")

    return y_true_total, y_pred_total


def evaluate_others(model, df_train, df_valid, model_name):
    df_train = df_train.copy()
    df_valid = df_valid.copy()

    _, _, df_x_valid, _ = prepare_dataset_per_sqm(df_train, df_valid)

    # 予測
    y_pred_log = model.predict(df_x_valid)

    # 対数から元の金額に戻す
    y_pred = np.expm1(y_pred_log)

    # 正解データ
    y_true = df_valid['money_room'].astype(float)

    # 実金額ベースでのMAPE計算
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"[{model_name}] MAPE: {mape * 100:.2f}% (件数: {len(y_true)})")

    return y_true, y_pred

def evaluate_models_tokyo_area_okinawa(train_df, valid_df, save_path, learned_iter):
    # モデルを分ける地域
    SPLITED_TOKYO = [11, 12, 13, 14] # 埼玉、千葉、東京、神奈川
    SPLITED_OKINAWA = [47] # 沖縄

    # 地域で分割
    # 東京近郊
    df_train_tokyo_area = train_df[train_df['addr1_1'].isin(SPLITED_TOKYO)]
    df_valid_tokyo_area = valid_df[valid_df['addr1_1'].isin(SPLITED_TOKYO)]
    # 沖縄
    df_train_okinawa = train_df[train_df['addr1_1'].isin(SPLITED_OKINAWA)]
    df_valid_okinawa = valid_df[valid_df['addr1_1'].isin(SPLITED_OKINAWA)]
    # それ以外
    df_train_other = train_df[~train_df['addr1_1'].isin(SPLITED_TOKYO + SPLITED_OKINAWA)]
    df_valid_other = valid_df[~valid_df['addr1_1'].isin(SPLITED_TOKYO + SPLITED_OKINAWA)]

    # 東京近郊
    df_train_tokyo_mansion, df_train_tokyo_others = split_by_type(df_train_tokyo_area)
    df_valid_tokyo_mansion, df_valid_tokyo_others = split_by_type(df_valid_tokyo_area)
    # 沖縄
    df_train_okinawa_mansion, df_train_okinawa_others = split_by_type(df_train_okinawa)
    df_valid_okinawa_mansion, df_valid_okinawa_others = split_by_type(df_valid_okinawa)
    # それ以外
    df_train_other_mansion, df_train_other_others = split_by_type(df_train_other)
    df_valid_other_mansion, df_valid_other_others = split_by_type(df_valid_other) 

    # 全体統合用のリスト
    all_y_true = []
    all_y_pred = []

    # --- 以下、各エリア・種別ごとにモデル実行 ---
    for area_name, train_m, test_m, train_o, test_o in [
        ("東京近郊", df_train_tokyo_mansion, df_valid_tokyo_mansion, df_train_tokyo_others, df_valid_tokyo_others),
        ("沖縄", df_train_okinawa_mansion, df_valid_okinawa_mansion, df_train_okinawa_others, df_valid_okinawa_others),
        ("その他地域", df_train_other_mansion, df_valid_other_mansion, df_train_other_others, df_valid_other_others)
    ]:
        print(f"\n{'='*60}\n【{area_name}】平米単価(Log+Fair) × 物件種別モデル\n{'='*60}")

        print("\n=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===")
        model_name = f"{area_name}_Mansion_PerSqm_LogFair_{learned_iter}"
        with open(f"{save_path}/{model_name}.pkl", "rb") as f:
            model = pickle.load(f)

        y_true_m, y_pred_m = evaluate_mansion(model, train_m, test_m, model_name)

        print("\n=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===")
        model_name = f"{area_name}_Others_Direct_LogFair_{learned_iter}"
        with open(f"{save_path}/{model_name}.pkl", "rb") as f:
            model = pickle.load(f)
        y_true_o, y_pred_o = evaluate_others(model, train_o, test_o, model_name)

        # 結果の統合（エリア内）
        if y_true_m is not None and y_true_o is not None:
            y_true_all = pd.concat([y_true_m, y_true_o])
            y_pred_all = np.concatenate([y_pred_m, y_pred_o])

            # 全体統合用に追加
            all_y_true.append(y_true_all)
            all_y_pred.append(y_pred_all)

            total_mape = mean_absolute_percentage_error(y_true_all, y_pred_all)
            print("\n" + "="*60)
            print(f"[{area_name}] 統合モデルの全体MAPE: {total_mape * 100:.2f}%")
            print(f"  - マンション件数: {len(y_true_m)}")
            print(f"  - 戸建・他件数: {len(y_true_o)}")
            print(f"  - 合計件数: {len(y_true_all)}")
            print("="*60)
        else:
            print(f"\n[警告] {area_name}でいずれかのモデルでデータ不足のため統合できませんでした")

    # --- 全エリア・全種別を統合した最終MAPE ---
    if len(all_y_true) > 0:
        final_y_true = pd.concat(all_y_true)
        final_y_pred = np.concatenate(all_y_pred)

        final_mape = mean_absolute_percentage_error(final_y_true, final_y_pred)

        print("\n" + "="*60)
        print("【最終結果】全エリア・全種別統合モデル")
        print("="*60)
        print(f"全体MAPE: {final_mape * 100:.2f}%")
        print(f"総件数: {len(final_y_true)}")
        print("="*60)


def evaluate_models_tokyo_kougai(train_df, valid_df, save_path, learned_iter):
    # モデルを分ける地域
    SPLITED_TOKYO = [13]  # 東京
    SPLITED_KOUGAI = [11, 12, 14] # 埼玉、千葉、神奈川
    SPLITED_OKINAWA = [47] # 沖縄

    # 地域で分割
    # 東京
    df_train_tokyo = train_df[train_df['addr1_1'].isin(SPLITED_TOKYO)]
    df_valid_tokyo = valid_df[valid_df['addr1_1'].isin(SPLITED_TOKYO)]
    # 郊外
    df_train_kougai = train_df[train_df['addr1_1'].isin(SPLITED_KOUGAI)]
    df_valid_kougai = valid_df[valid_df['addr1_1'].isin(SPLITED_KOUGAI)]

    # 物件種別で分割
    def split_by_type(df):
        is_mansion = df['building_type'].astype(str) == "マンション"
        return df[is_mansion].copy(), df[~is_mansion].copy()

    # 東京
    df_train_tokyo_mansion, df_train_tokyo_others = split_by_type(df_train_tokyo)
    df_valid_tokyo_mansion, df_valid_tokyo_others = split_by_type(df_valid_tokyo)
    # 郊外
    df_train_kougai_mansion, df_train_kougai_others = split_by_type(df_train_kougai)
    df_valid_kougai_mansion, df_valid_kougai_others = split_by_type(df_valid_kougai)

    # 全体統合用のリスト
    all_y_true = []
    all_y_pred = []

    # --- 以下、各エリア・種別ごとにモデル実行 ---
    for area_name, train_m, test_m, train_o, test_o in [
        ("東京", df_train_tokyo_mansion, df_valid_tokyo_mansion, df_train_tokyo_others, df_valid_tokyo_others),
        ("東京近郊", df_train_kougai_mansion, df_valid_kougai_mansion, df_train_kougai_others, df_valid_kougai_others),
    ]:
        print(f"\n{'='*60}\n【{area_name}】平米単価(Log+Fair) × 物件種別モデル\n{'='*60}")

        print("\n=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===")
        model_name = f"{area_name}_Mansion_PerSqm_LogFair_{learned_iter}"
        with open(f"{save_path}/{model_name}.pkl", "rb") as f:
            model = pickle.load(f)

        y_true_m, y_pred_m = evaluate_mansion(model, train_m, test_m, model_name)

        print("\n=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===")
        model_name = f"{area_name}_Others_Direct_LogFair_{learned_iter}"
        with open(f"{save_path}/{model_name}.pkl", "rb") as f:
            model = pickle.load(f)
        y_true_o, y_pred_o = evaluate_others(model, train_o, test_o, model_name)

        # 結果の統合（エリア内）
        if y_true_m is not None and y_true_o is not None:
            y_true_all = pd.concat([y_true_m, y_true_o])
            y_pred_all = np.concatenate([y_pred_m, y_pred_o])

            # 全体統合用に追加
            all_y_true.append(y_true_all)
            all_y_pred.append(y_pred_all)

            total_mape = mean_absolute_percentage_error(y_true_all, y_pred_all)
            print("\n" + "="*60)
            print(f"[{area_name}] 統合モデルの全体MAPE: {total_mape * 100:.2f}%")
            print(f"  - マンション件数: {len(y_true_m)}")
            print(f"  - 戸建・他件数: {len(y_true_o)}")
            print(f"  - 合計件数: {len(y_true_all)}")
            print("="*60)
        else:
            print(f"\n[警告] {area_name}でいずれかのモデルでデータ不足のため統合できませんでした")

    # --- 全エリア・全種別を統合した最終MAPE ---
    if len(all_y_true) > 0:
        final_y_true = pd.concat(all_y_true)
        final_y_pred = np.concatenate(all_y_pred)

        final_mape = mean_absolute_percentage_error(final_y_true, final_y_pred)

        print("\n" + "="*60)
        print("【最終結果】全エリア・全種別統合モデル")
        print("="*60)
        print(f"全体MAPE: {final_mape * 100:.2f}%")
        print(f"総件数: {len(final_y_true)}")
        print("="*60)

In [ ]:
save_path = "../model/0108"
learned_iter = 50000

evaluate_models_tokyo_area_okinawa(train_df, valid_df, save_path, learned_iter)


【東京近郊】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
[東京近郊_Mansion_PerSqm_LogFair_50000] MAPE: 11.70% (件数: 17015)

=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===
[東京近郊_Others_Direct_LogFair_50000] MAPE: 15.86% (件数: 10334)

[東京近郊] 統合モデルの全体MAPE: 13.28%
  - マンション件数: 17015
  - 戸建・他件数: 10334
  - 合計件数: 27349

【沖縄】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
[沖縄_Mansion_PerSqm_LogFair_50000] MAPE: 9.48% (件数: 429)

=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===
[沖縄_Others_Direct_LogFair_50000] MAPE: 17.66% (件数: 204)

[沖縄] 統合モデルの全体MAPE: 12.12%
  - マンション件数: 429
  - 戸建・他件数: 204
  - 合計件数: 633

【その他地域】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
[その他地域_Mansion_PerSqm_LogFair_50000] MAPE: 16.57% (件数: 15281)

=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===
[その他地域_Others_Direct_LogFair_50000] MAPE: 15.99% (件数: 19470)

[その他地域] 統合モデルの全体MAPE: 16.25%
  - マンション件数: 15281
  - 戸建・他件数: 19470
  - 合計件数: 34751

【最終結果】全エリア・全種別統合モデル
全体MAPE: 14.91%
総件数: 62733


In [31]:
save_path = "../model/0109"
learned_iter = 20000

evaluate_models_tokyo_area_okinawa(train_df, valid_df, save_path, learned_iter)


【東京近郊】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
[東京近郊_Mansion_PerSqm_LogFair_20000] MAPE: 11.60% (件数: 17015)

=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===
[東京近郊_Others_Direct_LogFair_20000] MAPE: 15.77% (件数: 10334)

[東京近郊] 統合モデルの全体MAPE: 13.18%
  - マンション件数: 17015
  - 戸建・他件数: 10334
  - 合計件数: 27349

【沖縄】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
[沖縄_Mansion_PerSqm_LogFair_20000] MAPE: 9.36% (件数: 429)

=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===
[沖縄_Others_Direct_LogFair_20000] MAPE: 17.51% (件数: 204)

[沖縄] 統合モデルの全体MAPE: 11.99%
  - マンション件数: 429
  - 戸建・他件数: 204
  - 合計件数: 633

【その他地域】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
[その他地域_Mansion_PerSqm_LogFair_20000] MAPE: 17.42% (件数: 15281)

=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===
[その他地域_Others_Direct_LogFair_20000] MAPE: 16.34% (件数: 19470)

[その他地域] 統合モデルの全体MAPE: 16.81%
  - マンション件数: 15281
  - 戸建・他件数: 19470
  - 合計件数: 34751

【最終結果】全エリア・全種別統合モデル
全体MAPE: 15.18%
総件数: 62733


In [32]:
save_path = "../model/0109_ver2"
learned_iter = 20000

evaluate_models_tokyo_kougai(train_df, valid_df, save_path, learned_iter)


【東京】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
[東京_Mansion_PerSqm_LogFair_20000] MAPE: 9.77% (件数: 8247)

=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===
[東京_Others_Direct_LogFair_20000] MAPE: 14.66% (件数: 2429)

[東京] 統合モデルの全体MAPE: 10.88%
  - マンション件数: 8247
  - 戸建・他件数: 2429
  - 合計件数: 10676

【東京近郊】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
[東京近郊_Mansion_PerSqm_LogFair_20000] MAPE: 13.46% (件数: 8768)

=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===
[東京近郊_Others_Direct_LogFair_20000] MAPE: 15.96% (件数: 7905)

[東京近郊] 統合モデルの全体MAPE: 14.65%
  - マンション件数: 8768
  - 戸建・他件数: 7905
  - 合計件数: 16673

【最終結果】全エリア・全種別統合モデル
全体MAPE: 13.18%
総件数: 27349


In [22]:
save_path = "../model/0107"
learned_iter = 10000

evaluate_models(train_df, valid_df, save_path, learned_iter)


【東京近郊】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
[東京近郊_Mansion_PerSqm_LogFair_10000] MAPE: 6.99% (件数: 17015)

=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===
[東京近郊_Others_Direct_LogFair_10000] MAPE: 7.50% (件数: 10334)

[東京近郊] 統合モデルの全体MAPE: 7.18%
  - マンション件数: 17015
  - 戸建・他件数: 10334
  - 合計件数: 27349

【沖縄】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
[沖縄_Mansion_PerSqm_LogFair_10000] MAPE: 3.81% (件数: 429)

=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===
[沖縄_Others_Direct_LogFair_10000] MAPE: 9.75% (件数: 204)

[沖縄] 統合モデルの全体MAPE: 5.72%
  - マンション件数: 429
  - 戸建・他件数: 204
  - 合計件数: 633

【その他地域】平米単価(Log+Fair) × 物件種別モデル

=== マンション(Type=1) - 平米単価モデル(Log+Fair) ===
[その他地域_Mansion_PerSqm_LogFair_10000] MAPE: 10.18% (件数: 15281)

=== 戸建・他(Type!=1) - 直接予測モデル(Log+Fair) ===
[その他地域_Others_Direct_LogFair_10000] MAPE: 10.07% (件数: 19470)

[その他地域] 統合モデルの全体MAPE: 10.12%
  - マンション件数: 15281
  - 戸建・他件数: 19470
  - 合計件数: 34751

【最終結果】全エリア・全種別統合モデル
全体MAPE: 8.79%
総件数: 62733


### モデルの確認

In [33]:
# faeture_importanceの可視化
output_file_path = "../model/0109"
filename = "東京近郊_Mansion_PerSqm_LogFair_20000.pkl"

with open(f"{output_file_path}/{filename}", "rb") as f:
    model = pickle.load(f)

df_x_train, _, _, _ = prepare_dataset_per_sqm(df_train_tokyo_mansion, df_valid_tokyo_mansion)
importance = pd.DataFrame(model.feature_importance(), index=df_x_train.columns, columns=['importance'])
importance.sort_values(by='importance', ascending=False, inplace=True)
importance.head(30)

,importance
money_kyoueki,37678
house_area,32803
year_built,32374
convenience_distance,28790
school_ele_distance,27937
school_jun_distance,25219
land_value_ratio_mean_5nn,22738
unit_area,21926
lon,21765
eki_name1,21307


In [34]:
# faeture_importanceの可視化
output_file_path = "../model/0109_ver2"
filename = "東京_Mansion_PerSqm_LogFair_20000.pkl"

with open(f"{output_file_path}/{filename}", "rb") as f:
    model = pickle.load(f)

df_x_train, _, _, _ = prepare_dataset_per_sqm(df_train_tokyo_mansion, df_valid_tokyo_mansion)
importance = pd.DataFrame(model.feature_importance(), index=df_x_train.columns, columns=['importance'])
importance.sort_values(by='importance', ascending=False, inplace=True)
importance.head(30)

,importance
money_kyoueki,17276
year_built,15734
house_area,13959
convenience_distance,13498
school_ele_distance,10807
lat,10609
unit_area,10536
land_value_ratio_mean_5nn,10196
total_passengers_1600m,9723
lon,9676


In [35]:
# faeture_importanceの可視化
output_file_path = "../model/0109_ver2"
filename = "東京近郊_Mansion_PerSqm_LogFair_20000.pkl"

with open(f"{output_file_path}/{filename}", "rb") as f:
    model = pickle.load(f)

df_x_train, _, _, _ = prepare_dataset_per_sqm(df_train_tokyo_mansion, df_valid_tokyo_mansion)
importance = pd.DataFrame(model.feature_importance(), index=df_x_train.columns, columns=['importance'])
importance.sort_values(by='importance', ascending=False, inplace=True)
importance.head(30)

,importance
money_kyoueki,25307
house_area,24130
year_built,21922
school_ele_distance,21849
school_jun_distance,19657
convenience_distance,19470
unit_area,15043
land_value_ratio_mean_5nn,14995
lon,14518
land_value_median_5nn,14231


In [12]:
importance.tail(30)

,importance
building_tag_id_防犯カメラ,1644
building_tag_id_バイク置き場あり,1641
building_tag_id_タイル貼り,1637
statuses_エレベーター,1614
building_tag_id_オートロック,1585
statuses_コンロ三口,1560
statuses_カウンターキッチン,1431
statuses_洗面所独立,1384
statuses_バルコニー,1363
building_structure,1282


In [14]:
# faeture_importanceの可視化
output_file_path = "../model/0107"
filename = "東京近郊_Others_Direct_LogFair_10000.pkl"

with open(f"{output_file_path}/{filename}", "rb") as f:
    model = pickle.load(f)

df_x_train, _, _, _ = prepare_dataset_direct(df_train_tokyo_others, df_valid_tokyo_others)
importance = pd.DataFrame(model.feature_importance(), index=df_x_train.columns, columns=['importance'])
importance.sort_values(by='importance', ascending=False, inplace=True)
importance.head(30)

,importance
house_area,21676
year_built,20702
school_ele_distance,19398
lon,18851
lat,18697
unit_area,18067
school_jun_distance,17851
convenience_distance,15826
walk_distance1,15420
eki_name1,15392


In [ ]:
importance.tail(30)

,importance
statuses_バス・トイレ別,1068
statuses_出窓,1041
building_structure,1003
statuses_ウォークインクローゼット,803
parking_distance,556
addr1_1,449
statuses_エアコン,379
statuses_床暖房,353
station_count_400m,332
room_kaisuu,269


In [18]:
# faeture_importanceの可視化
output_file_path = "../model/0107"
filename = "沖縄_Mansion_PerSqm_LogFair_10000.pkl"

with open(f"{output_file_path}/{filename}", "rb") as f:
    model = pickle.load(f)

df_x_train, _, _, _ = prepare_dataset_per_sqm(df_train_okinawa_mansion, df_valid_okinawa_mansion)
importance = pd.DataFrame(model.feature_importance(), index=df_x_train.columns, columns=['importance'])
importance.sort_values(by='importance', ascending=False, inplace=True)
importance.head(30)

,importance
year_built,73529
house_area,21030
money_kyoueki,19290
unit_area,17588
room_kaisuu,15011
school_ele_distance,14366
school_jun_distance,14144
walk_distance1,11711
lat,9522
lon,9498


In [20]:
# faeture_importanceの可視化
output_file_path = "../model/0107"
filename = "沖縄_Others_Direct_LogFair_10000.pkl"

with open(f"{output_file_path}/{filename}", "rb") as f:
    model = pickle.load(f)

df_x_train, _, _, _ = prepare_dataset_per_sqm(df_train_okinawa_others, df_valid_okinawa_others)
importance = pd.DataFrame(model.feature_importance(), index=df_x_train.columns, columns=['importance'])
importance.sort_values(by='importance', ascending=False, inplace=True)
importance.head(30)

,importance
year_built,93700
unit_area,24139
walk_distance1,20358
house_area,19688
school_ele_distance,17074
lon,16471
school_jun_distance,16446
lat,12855
land_kenpei,10560
madori_number_all,8460


In [ ]:
importance.tail(30)

,importance
building_tag_id_防犯カメラ,1644
building_tag_id_バイク置き場あり,1641
building_tag_id_タイル貼り,1637
statuses_エレベーター,1614
building_tag_id_オートロック,1585
statuses_コンロ三口,1560
statuses_カウンターキッチン,1431
statuses_洗面所独立,1384
statuses_バルコニー,1363
building_structure,1282


In [ ]:
# faeture_importanceの可視化
output_file_path = "../model/0107"
filename = "東京近郊_Others_Direct_LogFair_10000.pkl"

with open(f"{output_file_path}/{filename}", "rb") as f:
    model = pickle.load(f)

df_x_train, _, _, _ = prepare_dataset_direct(df_train_tokyo_others, df_valid_tokyo_others)
importance = pd.DataFrame(model.feature_importance(), index=df_x_train.columns, columns=['importance'])
importance.sort_values(by='importance', ascending=False, inplace=True)
importance.head(30)

,importance
house_area,21676
year_built,20702
school_ele_distance,19398
lon,18851
lat,18697
unit_area,18067
school_jun_distance,17851
convenience_distance,15826
walk_distance1,15420
eki_name1,15392


In [15]:
# faeture_importanceの可視化
output_file_path = "../model/0107"
filename = "その他地域_Mansion_PerSqm_LogFair_10000.pkl"

with open(f"{output_file_path}/{filename}", "rb") as f:
    model = pickle.load(f)

df_x_train, _, _, _ = prepare_dataset_per_sqm(df_train_other_mansion, df_valid_other_mansion)
importance = pd.DataFrame(model.feature_importance(), index=df_x_train.columns, columns=['importance'])
importance.sort_values(by='importance', ascending=False, inplace=True)
importance.head(30)

,importance
money_kyoueki,18526
school_ele_distance,16061
year_built,15848
house_area,15776
school_jun_distance,15692
eki_name1,13019
convenience_distance,12337
unit_area,11902
lat,11785
total_passengers_800m,11234


In [ ]:
importance.tail(30)

,importance
statuses_カウンターキッチン,1614
building_tag_id_防犯カメラ,1573
money_kyoueki_tax,1566
madori_kind_all,1523
statuses_エレベーター,1509
statuses_クローゼット,1473
statuses_追焚機能,1440
statuses_バス・トイレ別,1422
statuses_バルコニー,1350
statuses_コンロ三口,1304


In [16]:
# faeture_importanceの可視化
output_file_path = "../model/0107"
filename = "その他地域_Others_Direct_LogFair_10000.pkl"

with open(f"{output_file_path}/{filename}", "rb") as f:
    model = pickle.load(f)

df_x_train, _, _, _ = prepare_dataset_direct(df_train_other_others, df_valid_other_others)
importance = pd.DataFrame(model.feature_importance(), index=df_x_train.columns, columns=['importance'])
importance.sort_values(by='importance', ascending=False, inplace=True)
importance.head(30)

,importance
school_ele_distance,23124
school_jun_distance,22530
year_built,20632
house_area,20364
unit_area,17298
eki_name1,16157
lat,15777
walk_distance1,15425
convenience_distance,15031
lon,13613


### 予測

In [47]:
def prepare_dataset_per_sqm(df_train, df_test):
    # 平米単価の計算 (学習データのみ)
    df_train['house_area'] = df_train['house_area'].astype(float)
    df_test['house_area'] = df_test['house_area'].astype(float)

    # 学習データには money_room がある前提
    if 'price_per_sqm' not in df_train.columns:
        df_train['price_per_sqm'] = df_train['money_room'].astype(float) / df_train['house_area']

    # 説明変数の分離
    # target_ym, completion_year など不要なものは適宜除外してください
    drop_cols = ['money_room', 'price_per_sqm', 'target_ym']

    # カラムが存在する場合のみ削除
    train_drop = [c for c in drop_cols if c in df_train.columns]
    test_drop = [c for c in drop_cols if c in df_test.columns]

    df_x_train_all = df_train.drop(train_drop, axis=1).copy()
    df_x_test = df_test.drop(test_drop, axis=1).copy()

    # 目的変数を対数変換 (学習データのみ)
    df_y_train_all = np.log1p(df_train['price_per_sqm'].astype(float))

    # エンコード処理 (TrainとTestを合わせてFitさせる)
    label_encoder = LabelEncoder()
    encode_cols = list(set(df_x_train_all.columns.to_list()) - set(drop_cols) - set(NUMERICAL_FEATURES))

    for col in encode_cols:
        if col in df_x_train_all.columns:
            # TrainとTestの値を結合してFit（未知のカテゴリ対策）
            train_vals = df_x_train_all[col].astype(str)
            test_vals = df_x_test[col].astype(str)
            all_values = pd.concat([train_vals, test_vals])

            label_encoder.fit(all_values)
            df_x_train_all[col] = label_encoder.transform(train_vals)
            df_x_test[col] = label_encoder.transform(test_vals)

    # 数値変換
    for col in NUMERICAL_FEATURES:
        if col in df_x_train_all.columns:
            df_x_train_all[col] = df_x_train_all[col].astype(float)
            df_x_test[col] = df_x_test[col].astype(float)

    # カラム整合性確保
    common_cols = [c for c in df_x_train_all.columns if c in df_x_test.columns]
    df_x_train_all = df_x_train_all[common_cols]
    df_x_test = df_x_test[common_cols]

    return df_x_train_all, df_y_train_all, df_x_test


# --- 平米単価予測 + Log + Fair Loss モデル ---
def train_and_test_per_sqm_log_fair(df_train, df_test, params, num_iter, model_name="PerSqm_LogFair"):

    df_train = df_train.copy()
    df_test = df_test.copy()

    df_x_train_all, df_y_train_all, df_x_test = prepare_dataset_per_sqm(df_train, df_test)

    # df_testには正解がないため、df_trainを分割して検証データを作る
    # これによりEarly Stoppingが可能になります
    X_tr, X_val, y_tr, y_val = train_test_split(
        df_x_train_all, df_y_train_all, test_size=0.2, random_state=42
    )

    # LightGBMデータセット作成
    lgb_train = lgb.Dataset(X_tr, y_tr)
    lgb_eval = lgb.Dataset(X_val, y_val, reference=lgb_train)

    # 学習
    model = lgb.train(
        params,
        lgb_train,
        valid_sets=[lgb_train, lgb_eval],
        valid_names=['train', 'eval'],
        num_boost_round=num_iter,
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=True),lgb.log_evaluation(100)]
    )

    # 予測(対数空間の平米単価)
    y_pred_log_sqm = model.predict(df_x_test)

    # 対数から元の平米単価に戻す
    y_pred_sqm = np.expm1(y_pred_log_sqm)

    # 平米単価 × 面積 で総額に変換
    y_pred_total = y_pred_sqm * df_x_test['house_area']

    # df_testにmoney_roomがあれば精度確認、なければ予測値のみ返す
    if 'money_room' in df_test.columns:
        y_true_total = df_test['money_room'].astype(float)
        mape = mean_absolute_percentage_error(y_true_total, y_pred_total)
        print(f"[{model_name}] (Test Set Existing) MAPE: {mape * 100:.2f}%")
        return model, y_true_total, y_pred_total
    else:
        print(f"[{model_name}] 予測完了 ")
        return model, None, y_pred_total


def prepare_dataset_direct(df_train, df_test):
    # 説明変数の分離
    drop_cols = ['money_room', 'target_ym', 'price_per_sqm']

    train_drop = [c for c in drop_cols if c in df_train.columns]
    test_drop = [c for c in drop_cols if c in df_test.columns]

    df_x_train_all = df_train.drop(train_drop, axis=1).copy()
    df_x_test = df_test.drop(test_drop, axis=1).copy()

    # 目的変数を対数変換
    df_y_train_all = np.log1p(df_train['money_room'].astype(float))

    # エンコード処理
    label_encoder = LabelEncoder()
    encode_cols = list(set(df_x_train_all.columns.to_list()) - set(drop_cols) - set(NUMERICAL_FEATURES))

    for col in encode_cols:
        if col in df_x_train_all.columns:
            train_vals = df_x_train_all[col].astype(str)
            test_vals = df_x_test[col].astype(str)
            all_values = pd.concat([train_vals, test_vals])

            label_encoder.fit(all_values)
            df_x_train_all[col] = label_encoder.transform(train_vals)
            df_x_test[col] = label_encoder.transform(test_vals)

    # 数値変換
    for col in NUMERICAL_FEATURES:
        if col in df_x_train_all.columns:
            df_x_train_all[col] = df_x_train_all[col].astype(float)
            df_x_test[col] = df_x_test[col].astype(float)

    # カラムの整合性確保
    common_cols = [c for c in df_x_train_all.columns if c in df_x_test.columns]
    df_x_train_all = df_x_train_all[common_cols]
    df_x_test = df_x_test[common_cols]

    return df_x_train_all, df_y_train_all, df_x_test


# --- 直接価格予測 + Log + Fair Loss モデル ---
def train_and_test_direct_log_fair(df_train, df_test, params, num_iter, model_name="Direct_LogFair"):
    df_train = df_train.copy()
    df_test = df_test.copy()

    df_x_train_all, df_y_train_all, df_x_test = prepare_dataset_direct(df_train, df_test)

    # 【重要】検証用データの作成 (df_trainを分割)
    X_tr, X_val, y_tr, y_val = train_test_split(
        df_x_train_all, df_y_train_all, test_size=0.2, random_state=42
    )

    # LightGBMデータセット作成
    lgb_train = lgb.Dataset(X_tr, y_tr)
    lgb_eval = lgb.Dataset(X_val, y_val, reference=lgb_train)

    # 学習
    model = lgb.train(
        params,
        lgb_train,
        valid_sets=[lgb_train, lgb_eval],
        valid_names=['train', 'eval'],
        num_boost_round=num_iter,
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=True),lgb.log_evaluation(100)]
    )

    # 予測(対数空間)
    y_pred_log = model.predict(df_x_test)

    # 対数から元の金額に戻す
    y_pred = np.expm1(y_pred_log)

    # 結果の返却
    if 'money_room' in df_test.columns:
        y_true = df_test['money_room'].astype(float)
        mape = mean_absolute_percentage_error(y_true, y_pred)
        print(f"[{model_name}] (Test Set Existing) MAPE: {mape * 100:.2f}%")
        return model, y_true, y_pred
    else:
        print(f"[{model_name}] 予測完了 ")
        return model, None, y_pred

In [40]:
# testデータ読み込み
test_df = pd.read_pickle(f"{ROOT_DIR}/0109_preprocessed_test_df.pkl")

test_df.drop(["bukken_id"], axis=1, inplace=True)
test_df.head()

,addr1_1,addr1_2,building_type,building_structure,convenience_distance,drugstore_distance,eki_name1,eki_name2,floor_count,house_area,...,walk_distance2,target_y,madori_str,post_number,station_count_800m,total_passengers_800m,station_count_1600m,total_passengers_1600m,land_value_median_5nn,land_value_ratio_mean_5nn
0,24,205,マンション,SRC,474.0,118.0,桑名,桑名,14,70,...,909,2023,3LDK,511-0002,3,31505.0,5,35716.0,85900.0,0.76
1,24,205,一戸建,木造,NaN,NaN,馬道,NA,2,171,...,9999,2023,6LDK,511-0814,1,547.0,8,38738.0,88000.0,1.42
2,23,224,一戸建,軽量鉄骨,650.0,NaN,大野町,NA,2,78,...,9999,2023,3LDK,478-0024,0,0.0,0,0.0,56100.0,0.16
3,23,224,一戸建,木造,460.0,1250.0,寺本,NA,1,93,...,9999,2023,4LDK,478-0001,1,3988.0,2,8652.0,71000.0,3.20
4,24,205,一戸建,木造,522.0,NaN,星川,七和,2,105,...,2400,2023,4LDK,511-0935,0,0.0,1,1077.0,41600.0,0.46


In [39]:
all_train_df.head()

,addr1_1,addr1_2,building_type,building_structure,convenience_distance,drugstore_distance,eki_name1,eki_name2,floor_count,house_area,...,money_room,target_y,madori_str,post_number,station_count_800m,total_passengers_800m,station_count_1600m,total_passengers_1600m,land_value_median_5nn,land_value_ratio_mean_5nn
0,24,205,一戸建,木造,NaN,NaN,在良,NA,2,106.82,...,13980000,2019,4LDK,511-0932,0,0.0,1,231.0,41300.0,-0.20
1,24,205,一戸建,軽量鉄骨,NaN,NaN,星川,NA,2,134.04,...,24480000,2019,4LDK,511-0902,0,0.0,2,1308.0,57500.0,0.32
2,24,205,一戸建,木造,NaN,NaN,蓮花寺,NA,2,114.59,...,24480000,2019,4LDK,511-0902,0,0.0,3,1848.0,63400.0,0.26
3,23,224,一戸建,木造,NaN,NaN,寺本,尾張横須賀,2,106.81,...,16300000,2019,3LDK,478-0001,1,3988.0,3,15129.0,66700.0,0.96
4,23,224,マンション,RC,NaN,1060.0,寺本,朝倉,6,76.74,...,18800000,2019,3LDK,478-0001,1,3988.0,3,15129.0,66700.0,0.96


In [41]:
# モデル保存先の作成
SAVE_PATH = "../model/0109_pred"
os.makedirs(SAVE_PATH, exist_ok=True)

In [48]:
# Fair Lossを使用したパラメータ設定
PARAMS = {
    'objective': 'fair',      # Fair Loss
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'fair_c': 0.1,
    'verbose': -1,
    'random_state': 42,
    "learning_rate": 0.1
}
NUM_ITER = 10000

In [42]:
# モデルを分ける地域
SPLITED_TOKYO = [11, 12, 13, 14] # 埼玉、千葉、東京、神奈川
SPLITED_OKINAWA = [47] # 沖縄

# # モデルを分ける建物
# SPLITED_BUILDING1 = "マンション"
# SPLITED_BUILDING2 = "一軒家"

In [ ]:
from sklearn.model_selection import KFold

N_SPLITS = 3
kfolds = KFold(n_splits=N_SPLITS, shuffle=True, random_state=8)

all_submit_df = []
for i, (train_cv_no, eval_cv_no) in enumerate(kfolds.split(all_train_df)):
    # 地域で分割
    # 東京近郊
    df_train_tokyo_area = all_train_df[all_train_df['addr1_1'].isin(SPLITED_TOKYO)]
    df_test_tokyo_area = test_df[test_df['addr1_1'].isin(SPLITED_TOKYO)]
    # 沖縄
    df_train_okinawa = all_train_df[all_train_df['addr1_1'].isin(SPLITED_OKINAWA)]
    df_test_okinawa = test_df[test_df['addr1_1'].isin(SPLITED_OKINAWA)]
    # それ以外
    df_train_other = all_train_df[~all_train_df['addr1_1'].isin(SPLITED_TOKYO + SPLITED_OKINAWA)]
    df_test_other = test_df[~test_df['addr1_1'].isin(SPLITED_TOKYO + SPLITED_OKINAWA)]
    # 物件種別で分割
    def split_by_type(df):
        is_mansion = df['building_type'].astype(str) == "マンション"
        return df[is_mansion].copy(), df[~is_mansion].copy()

    # 東京近郊
    df_train_tokyo_mansion, df_train_tokyo_others = split_by_type(df_train_tokyo_area)
    df_test_tokyo_mansion, df_test_tokyo_others = split_by_type(df_test_tokyo_area)
    # 沖縄
    df_train_okinawa_mansion, df_train_okinawa_others = split_by_type(df_train_okinawa)
    df_test_okinawa_mansion, df_test_okinawa_others = split_by_type(df_test_okinawa)
    # それ以外
    df_train_other_mansion, df_train_other_others = split_by_type(df_train_other)
    df_test_other_mansion, df_test_other_others = split_by_type(df_test_other) 

    # --- 予測実行と結果の集約 ---
    prediction_results = [] # DataFrameを格納するリスト

    for area_name, train_m, test_m, train_o, test_o in [
        ("東京近郊", df_train_tokyo_mansion, df_test_tokyo_mansion, df_train_tokyo_others, df_test_tokyo_others),
        ("沖縄", df_train_okinawa_mansion, df_test_okinawa_mansion, df_train_okinawa_others, df_test_okinawa_others),
        ("その他地域", df_train_other_mansion, df_test_other_mansion, df_train_other_others, df_test_other_others)
    ]:
        print(f"\n{'='*60}\n【{area_name}】平米単価(Log+Fair) × 物件種別モデル\n{'='*60}")

        # --- マンション予測 ---
        model_name = f"{area_name}_Mansion_PerSqm_LogFair_{NUM_ITER}"
        model, _, y_pred_m = train_and_test_per_sqm_log_fair(
            train_m, test_m, PARAMS, NUM_ITER, model_name=model_name
        )
        # モデルの保存
        with open(f"{SAVE_PATH}/{model_name}.pkl", "wb") as f:
            pickle.dump(model, f)

        # 【修正点】test_m.index (元の行番号) をインデックスとして使用
        if y_pred_m is not None and len(test_m) > 0:
            temp_df_m = pd.DataFrame({
                'money_room': y_pred_m
            }, index=test_m.index)
            prediction_results.append(temp_df_m)


        # --- 戸建・他予測 ---
        model_name = f"{area_name}_Others_Direct_LogFair_{NUM_ITER}" 
        model, _, y_pred_o = train_and_test_direct_log_fair(
            train_o, test_o, params=PARAMS, num_iter=NUM_ITER, model_name=model_name
        )
        # モデルの保存
        with open(f"{SAVE_PATH}/{model_name}_fold_{i}.pkl", "wb") as f:
            pickle.dump(model, f)

        # test_o.index (元の行番号) をインデックスとして使用
        if y_pred_o is not None and len(test_o) > 0:
            temp_df_o = pd.DataFrame({
                'money_room': y_pred_o
            }, index=test_o.index)
            prediction_results.append(temp_df_o)


    # --- 提出用データの作成 ---

    # 1. 全ての分割データの予測結果を縦に結合
    submit_df = pd.concat(prediction_results, axis=0)

    # 2. インデックス（元の行番号 0, 1, 2...）順に並び替え
    # これにより、test.csvの元の並び順に戻ります
    submit_df = submit_df.sort_index()

    # 3. インデックスをカラム('id')として取り出す
    submit_df = submit_df.reset_index()
    submit_df.columns = ['id', f'money_room_fold_{i}']

    # サンプル(000000)に合わせてIDを6桁ゼロ埋めする場合
    submit_df['id'] = submit_df['id'].apply(lambda x: '{:06}'.format(x))
    all_submit_df.append(submit_df)

all_submit_df = pd.concat(all_submit_df, axis=1)
# foldごとの予測結果を平均化
money_room_cols = [col for col in all_submit_df.columns if col.startswith('money_room_fold_')]
all_submit_df['money_room'] = all_submit_df[money_room_cols].mean(axis=1)
submit_df = all_submit_df[['id', 'money_room']] 
submit_df = submit_df.iloc[:, [-2, -1]]
# 4. CSV出力 (header=False, index=False)
submit_df.to_csv(f"{SAVE_PATH}/submit.csv", index=False, header=False)



【東京近郊】平米単価(Log+Fair) × 物件種別モデル
Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 0.19223	eval's rmse: 0.197747
[200]	train's rmse: 0.173283	eval's rmse: 0.183089
[300]	train's rmse: 0.163067	eval's rmse: 0.1773
[400]	train's rmse: 0.155672	eval's rmse: 0.173644
[500]	train's rmse: 0.149729	eval's rmse: 0.171114
[600]	train's rmse: 0.144541	eval's rmse: 0.169124
[700]	train's rmse: 0.140091	eval's rmse: 0.167492
[800]	train's rmse: 0.135918	eval's rmse: 0.166267
[900]	train's rmse: 0.132268	eval's rmse: 0.165315
[1000]	train's rmse: 0.128846	eval's rmse: 0.164329
[1100]	train's rmse: 0.125906	eval's rmse: 0.163638
[1200]	train's rmse: 0.123153	eval's rmse: 0.162907
[1300]	train's rmse: 0.120505	eval's rmse: 0.162202
[1400]	train's rmse: 0.118053	eval's rmse: 0.16164
[1500]	train's rmse: 0.115826	eval's rmse: 0.161247
[1600]	train's rmse: 0.11365	eval's rmse: 0.160803
[1700]	train's rmse: 0.111704	eval's rmse: 0.160402
[1800]	train's rmse: 0.10976	eval's 

In [ ]:
# モデルを分ける地域
SPLITED_TOKYO = [13] # 東京
SPLITED_KOUGAI = [11, 12, 14] # 埼玉、千葉、神奈川

# # モデルを分ける建物
# SPLITED_BUILDING1 = "マンション"
# SPLITED_BUILDING2 = "一軒家"

In [ ]:
from sklearn.model_selection import KFold

N_SPLITS = 4
kfolds = KFold(n_splits=N_SPLITS, shuffle=True, random_state=8)

all_submit_df = []
for i, (train_cv_no, eval_cv_no) in enumerate(kfolds.split(all_train_df)):
    # 地域で分割
    # 東京近郊
    df_train_tokyo_area = all_train_df[all_train_df['addr1_1'].isin(SPLITED_TOKYO)]
    df_test_tokyo_area = test_df[test_df['addr1_1'].isin(SPLITED_TOKYO)]
    # 沖縄
    df_train_okinawa = all_train_df[all_train_df['addr1_1'].isin(SPLITED_OKINAWA)]
    df_test_okinawa = test_df[test_df['addr1_1'].isin(SPLITED_OKINAWA)]
    # それ以外
    df_train_other = all_train_df[~all_train_df['addr1_1'].isin(SPLITED_TOKYO + SPLITED_OKINAWA)]
    df_test_other = test_df[~test_df['addr1_1'].isin(SPLITED_TOKYO + SPLITED_OKINAWA)]
    # 物件種別で分割
    def split_by_type(df):
        is_mansion = df['building_type'].astype(str) == "マンション"
        return df[is_mansion].copy(), df[~is_mansion].copy()

    # 東京近郊
    df_train_tokyo_mansion, df_train_tokyo_others = split_by_type(df_train_tokyo_area)
    df_test_tokyo_mansion, df_test_tokyo_others = split_by_type(df_test_tokyo_area)
    # 沖縄
    df_train_okinawa_mansion, df_train_okinawa_others = split_by_type(df_train_okinawa)
    df_test_okinawa_mansion, df_test_okinawa_others = split_by_type(df_test_okinawa)
    # それ以外
    df_train_other_mansion, df_train_other_others = split_by_type(df_train_other)
    df_test_other_mansion, df_test_other_others = split_by_type(df_test_other) 

    # --- 予測実行と結果の集約 ---
    prediction_results = [] # DataFrameを格納するリスト

    for area_name, train_m, test_m, train_o, test_o in [
        ("東京近郊", df_train_tokyo_mansion, df_test_tokyo_mansion, df_train_tokyo_others, df_test_tokyo_others),
        ("沖縄", df_train_okinawa_mansion, df_test_okinawa_mansion, df_train_okinawa_others, df_test_okinawa_others),
        ("その他地域", df_train_other_mansion, df_test_other_mansion, df_train_other_others, df_test_other_others)
    ]:
        print(f"\n{'='*60}\n【{area_name}】平米単価(Log+Fair) × 物件種別モデル\n{'='*60}")

        # --- マンション予測 ---
        model_name = f"{area_name}_Mansion_PerSqm_LogFair_{NUM_ITER}"
        model, _, y_pred_m = train_and_test_per_sqm_log_fair(
            train_m, test_m, PARAMS, NUM_ITER, model_name=model_name
        )
        # モデルの保存
        with open(f"{SAVE_PATH}/{model_name}.pkl", "wb") as f:
            pickle.dump(model, f)

        # 【修正点】test_m.index (元の行番号) をインデックスとして使用
        if y_pred_m is not None and len(test_m) > 0:
            temp_df_m = pd.DataFrame({
                'money_room': y_pred_m
            }, index=test_m.index)
            prediction_results.append(temp_df_m)


        # --- 戸建・他予測 ---
        model_name = f"{area_name}_Others_Direct_LogFair_{NUM_ITER}" 
        model, _, y_pred_o = train_and_test_direct_log_fair(
            train_o, test_o, params=PARAMS, num_iter=NUM_ITER, model_name=model_name
        )
        # モデルの保存
        with open(f"{SAVE_PATH}/{model_name}_fold_{i}.pkl", "wb") as f:
            pickle.dump(model, f)

        # test_o.index (元の行番号) をインデックスとして使用
        if y_pred_o is not None and len(test_o) > 0:
            temp_df_o = pd.DataFrame({
                'money_room': y_pred_o
            }, index=test_o.index)
            prediction_results.append(temp_df_o)


    # --- 提出用データの作成 ---

    # 1. 全ての分割データの予測結果を縦に結合
    submit_df = pd.concat(prediction_results, axis=0)

    # 2. インデックス（元の行番号 0, 1, 2...）順に並び替え
    # これにより、test.csvの元の並び順に戻ります
    submit_df = submit_df.sort_index()

    # 3. インデックスをカラム('id')として取り出す
    submit_df = submit_df.reset_index()
    submit_df.columns = ['id', f'money_room_fold_{i}']

    # サンプル(000000)に合わせてIDを6桁ゼロ埋めする場合
    submit_df['id'] = submit_df['id'].apply(lambda x: '{:06}'.format(x))
    all_submit_df.append(submit_df)


all_submit_df = pd.concat(all_submit_df, axis=1)
# foldごとの予測結果を平均化
money_room_cols = [col for col in all_submit_df.columns if col.startswith('money_room_fold_')]
all_submit_df['money_room'] = all_submit_df[money_room_cols].mean(axis=1)
submit_df = all_submit_df[['id', 'money_room']] 
# 4. CSV出力 (header=False, index=False)
submit_df.to_csv(f"{SAVE_PATH}/submit.csv", index=False, header=False)



【東京近郊】平米単価(Log+Fair) × 物件種別モデル
Training until validation scores don't improve for 500 rounds
[500]	train's rmse: 0.162849	eval's rmse: 0.173514
[1000]	train's rmse: 0.145888	eval's rmse: 0.165634
[1500]	train's rmse: 0.13507	eval's rmse: 0.161999
[2000]	train's rmse: 0.126562	eval's rmse: 0.159498
[2500]	train's rmse: 0.119763	eval's rmse: 0.157849
[3000]	train's rmse: 0.113973	eval's rmse: 0.156517
[3500]	train's rmse: 0.109051	eval's rmse: 0.155557
[4000]	train's rmse: 0.104689	eval's rmse: 0.154815
[4500]	train's rmse: 0.100753	eval's rmse: 0.154161
[5000]	train's rmse: 0.0971501	eval's rmse: 0.153672
[5500]	train's rmse: 0.0939907	eval's rmse: 0.153296
[6000]	train's rmse: 0.0910349	eval's rmse: 0.152915
[6500]	train's rmse: 0.0883699	eval's rmse: 0.152632
[7000]	train's rmse: 0.0858164	eval's rmse: 0.152302
[7500]	train's rmse: 0.0834847	eval's rmse: 0.152029
[8000]	train's rmse: 0.0813565	eval's rmse: 0.151843
[8500]	train's rmse: 0.0793052	eval's rmse: 0.151689
[9000]	train's r